### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [2]:
from unsloth import FastLanguageModel
import torch

dtype = ( None )
load_in_4bit = False
load_in_8bit = False
#unsloth/Llama-3.2-3B-Instruct
#Qwen/Qwen2.5-Coder-3B
#Qwen/Qwen2.5-Coder-7B-Instruct
#Qwen/Qwen2.5-14B-Instruct
#unsloth/Qwen3-4B

######---Parameters to change---#######
base_model="Qwen3-4B"      
max_seq_length = 2048
Rank=256
sample_len=15000
max_iter_steps=2000
###--------------------------------###

train_parameters=f"_lora_fp16_r{Rank}_s{sample_len}_i{max_iter_steps}_msl{max_seq_length}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/"+base_model,
    #model_name= "./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i2000_msl2048",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit
)

print(model.dtype)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-05 18:25:27 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3. vLLM: 0.8.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

torch.bfloat16


In [3]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = Rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 123,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.4.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [5]:
df_train1=pd.read_csv('./drawing-with-llms/train_filtered_1_batch_gpt4.csv')
df_train2=pd.read_csv('./drawing-with-llms/train_filtered_2_batch_deepseek_v3.csv')
df_train3=pd.read_csv('./drawing-with-llms/train_filtered_3_batch_gemini_2o.csv')
df_train4=pd.read_csv('./drawing-with-llms/train_master_1_batch_gemini_15_1st10k.csv')
df_train5=pd.read_csv('./drawing-with-llms/train_master_2_batch_gemini_20_2nd10k.csv')
df_train6=pd.read_csv('./drawing-with-llms/train_master_3_batch_gemini_20_3rd28k.csv')

In [6]:
df_train1=df_train1[['description','clean_svg']]
df_train1.shape

(1975, 2)

In [7]:
df_train2=df_train2[['description','clean_svg']]
df_train2.shape

(1716, 2)

In [8]:
df_train3=df_train3[['description','clean_svg']]
df_train3.shape

(1355, 2)

In [9]:
df_train4=df_train4[df_train4['gemini_sl_score'] > 0.5]
df_train4=df_train4[['description','extracted_svg']]
df_train4.columns=['description','clean_svg']
df_train4.shape

(2119, 2)

In [10]:
df_train5=df_train5[df_train5['gemini_sl_score'] > 0.5]
df_train5=df_train5[['description','extracted_svg']]
df_train5.columns=['description','clean_svg']
df_train5.shape

(3219, 2)

In [11]:
df_train6=df_train6[df_train6['gemini_cleaned_svg_sl_score'] > 0.5]
df_train6=df_train6[['description','cleaned_svg']]
df_train6.columns=['description','clean_svg']
df_train6.shape

(5710, 2)

In [12]:
df_train=pd.concat([df_train1,df_train2,df_train3,df_train4,df_train5,df_train6],axis=0)
df_train.shape

(16094, 2)

In [13]:
df_test1=pd.read_csv('./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv')
df_test2=pd.read_csv('./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv')

In [14]:
df_test1=df_test1[['description','clean_svg']]
df_test1.shape

(75, 2)

In [15]:
df_test2=df_test2[['description','clean_svg']]
df_test2.shape

(75, 2)

In [16]:
df_test=pd.concat([df_test1,df_test2],axis=0)
df_test.shape

(150, 2)

In [17]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["description"]  # Using 'topic' as instruction
    svgs = examples["clean_svg"]  # Using 'svg_code' as output
    texts = []

    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        text = alpaca_prompt.format(f"Generate a SVG code for the given input:",topic,svg_code) + EOS_TOKEN
        texts.append(text)
       
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset_train = Dataset.from_pandas(df_train)
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)

dataset_test = Dataset.from_pandas(df_test)
dataset_test = dataset_test.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/16094 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [18]:
# Check dataset sample output
dataset_train['text'][0]

'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate a SVG code for the given input:\n\n### Input:\n\'Golden wheat fields under a setting sun\',\n\n### Response:\n<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n  <!-- Background for the sky -->\n  <rect x="0" y="0" width="200" height="100" fill="orange" opacity="0.7"/>\n  \n  <!-- Sun -->\n  <circle cx="100" cy="50" r="30" fill="yellow" opacity="0.8"/>\n  \n  <!-- Wheat fields -->\n  <rect x="0" y="100" width="200" height="100" fill="goldenrod"/>\n  \n  <!-- Wheat stalks -->\n  <g stroke="saddlebrown" stroke-width="2">\n    <line x1="30" y1="100" x2="30" y2="150"/>\n    <line x1="50" y1="100" x2="50" y2="150"/>\n    <line x1="70" y1="100" x2="70" y2="150"/>\n    <line x1="90" y1="100" x2="90" y2="150"/>\n    <line x1="110" y1="100" x2="110" y2="150"/>\n    <line

In [19]:
import mlflow
mlflow.set_tracking_uri("ml_unsloth_runs")  # Or your remote tracking URI
mlflow.set_experiment("unsloth-lora-experiments")  # Your experiment name

<Experiment: artifact_location='/home/ml_unsloth_runs/990149045631854310', creation_time=1745245436792, experiment_id='990149045631854310', last_update_time=1745245436792, lifecycle_stage='active', name='unsloth-lora-experiments', tags={}>

In [20]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test,  # Add test dataset here
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = max_iter_steps,
        learning_rate = 5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit", # "adamw_torch" better for fp16
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        eval_strategy = "steps", 
        eval_steps = 100, 
        output_dir = "outputs",
        report_to = "mlflow", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/16094 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

In [21]:
formatted_model = base_model.replace(".", "").replace("-", "_")
run_name=formatted_model+train_parameters
with mlflow.start_run(run_name=run_name):
    trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,094 | Num Epochs = 2 | Total steps = 2,000
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 528,482,304/4,550,950,400 (11.61% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.193800,0.226092
200,0.187000,0.219687
300,0.182900,0.199726
400,0.167100,0.192151
500,0.172400,0.192055
600,0.165400,0.190732
700,0.156100,0.184990
800,0.154700,0.188955
900,0.159700,0.183679
1000,0.146600,0.180941


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [22]:
# #This ONLY saves the LoRA adapters, and not the full model.
# model.save_pretrained("./lora/lora_model_3b_v3") # Local saving
# tokenizer.save_pretrained("./lora/lora_model_3b_v3")

In [23]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [24]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [25]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
# model_name = "./lora/lora_model_3b_v2", # YOUR MODEL YOU USED FOR TRAINING
# max_seq_length = 2048,
# dtype = (None),
# load_in_4bit = False,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference


In [26]:
# # alpaca_prompt = You MUST copy from above!
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Please write a SVG code fo rthe given topic?", # instruction
#         "Golden sun rising in the east", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
# tokenizer.batch_decode(outputs)

In [27]:
#save merged 16bit
import os
dir_path = "./lora/"+run_name
os.makedirs(dir_path, exist_ok=True)
model.save_pretrained_merged(dir_path, tokenizer, save_method = "merged_16bit")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 11.54 out of 31.21 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 11%|████▉                                       | 4/36 [00:00<00:00, 39.05it/s]
We will save to Disk and not RAM now.
100%|███████████████████████████████████████████| 36/36 [00:05<00:00,  6.37it/s]


Unsloth: Saving tokenizer... Done.
Done.
